### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="amex_iid",
    dataset_year="2022",
    domain_str="finance",
    # Data Source
    dataset_source="Kaggle",
    original_dataset_source_download_link="https://www.kaggle.com/competitions/amex-default-prediction/",
    download_description="""
We start with the preprocessed version of the dataset in Parquet format provided by
the user 'raddar' on Kaggle. At the root of this project, I ran on linux with
the kaggle CLI installed and authenticated:

kaggle datasets download -d raddar/amex-data-integer-dtypes-parquet-format --unzip -p amex_dataset && rm amex_dataset/test.parquet
kaggle competitions download -c amex-default-prediction -f train_labels.csv -p amex_dataset && cd amex_dataset && unzip train_labels.csv.zip train_labels.csv && rm train_labels.csv.zip && cd ..
mkdir -p local-data-warehouse/amex_iid && mv amex_dataset/* local-data-warehouse/amex_iid/ && rm -rf amex_dataset
""",
    # References
    academic_reference_bibtex="""@misc{howard2022amex,
  author       = {Howard, Addison and AritraAmex and Xu, Di and Vashani, Hossein and inversion and Negin and Dane, Sohier},
  title        = {American Express -- Default Prediction},
  year         = {2022},
  howpublished = {Kaggle Competition},
  url          = {https://kaggle.com/competitions/amex-default-prediction},
  note         = {Accessed via Kaggle}
}
""",
    academic_reference_bibtex_key="howard2022amex",
    licence="Kaggle Competition License",
    data_tags=["IID"],
    curation_comments="""
Our preprocessing followed two Kaggle notebooks (https://www.kaggle.com/code/cdeotte/xgboost-starter-0-793,
https://www.kaggle.com/code/huseyincot/amex-agg-data-how-it-created), which follows
some of the most common steps used by top competitors in this competition. This final
version of the data becomes IID at the customer_ID level and drops temporal information.

- We transformed the "S_2" column to datetime format. It is used to sort the data for
 groupby aggregations based on the last value working correctly. Then it is dropped.
- We filled missing values with -127 (to be able to cast to int types to reduce memory).
- We transformed the list of transactions per customer to a single row per customer by
 aggregating numerical features with mean, std, min, max, and last value using groupby.
 Categorical features were aggregated with count, last value, and nunique.
- We drop the customer index and shuffled the data.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="target",
    problem_type="binary_classification",
    objective_metric_name="amex_metric",
    stratify_on="target",
)

## Preprocessing

In [2]:
import pandas as pd

df = pd.read_parquet(dataset_mold.path / "train.parquet")
print("Loaded data shape:", df.shape)

df.S_2 = pd.to_datetime(df.S_2)
df = df.sort_values(["customer_ID", "S_2"])

df = df.fillna(-127)

# Feature Engineering
all_cols = [c for c in list(df.columns) if c not in ["customer_ID", "S_2"]]
cat_features = [
    "B_30",
    "B_38",
    "D_114",
    "D_116",
    "D_117",
    "D_120",
    "D_126",
    "D_63",
    "D_64",
    "D_66",
    "D_68",
]
num_features = [col for col in all_cols if col not in cat_features]

num_agg = df.groupby("customer_ID")[num_features].agg(
    ["mean", "std", "min", "max", "last"]
)
num_agg.columns = ["_".join(x) for x in num_agg.columns]

cat_agg = df.groupby("customer_ID")[cat_features].agg(["count", "last", "nunique"])
cat_agg.columns = ["_".join(x) for x in cat_agg.columns]

df = pd.concat([num_agg, cat_agg], axis=1)
del num_agg, cat_agg

# Add labels
targets = pd.read_csv(dataset_mold.path / "train_labels.csv")
targets = targets.set_index("customer_ID")
df = df.merge(targets, left_index=True, right_index=True, how="left")
df.target = df.target.astype("int8")
del targets

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

Loaded data shape: (5531451, 190)


## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False,
)


#### Dataset Overview
Rows: 458,913
Columns: 919

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,P_2_mean,P_2_std,P_2_min,P_2_max,P_2_last,D_39_mean,D_39_std,D_39_min,D_39_max,D_39_last,B_1_mean,B_1_std,B_1_min,B_1_max,B_1_last,B_2_mean,B_2_std,B_2_min,B_2_max,B_2_last,R_1_mean,R_1_std,R_1_min,R_1_max,R_1_last,S_3_mean,S_3_std,S_3_min,S_3_max,S_3_last,D_41_mean,D_41_std,D_41_min,D_41_max,D_41_last,B_3_mean,B_3_std,B_3_min,B_3_max,B_3_last,D_42_mean,D_42_std,D_42_min,D_42_max,D_42_last,D_43_mean,D_43_std,D_43_min,D_43_max,D_43_last,D_44_mean,D_44_std,D_44_min,D_44_max,D_44_last,B_4_mean,B_4_std,B_4_min,B_4_max,B_4_last,D_45_mean,D_45_std,D_45_min,D_45_max,D_45_last,B_5_mean,B_5_std,B_5_min,B_5_max,B_5_last,R_2_mean,R_2_std,R_2_min,R_2_max,R_2_last,D_46_mean,D_46_std,D_46_min,D_46_max,D_46_last,D_47_mean,D_47_std,D_47_min,D_47_max,D_47_last,D_48_mean,D_48_std,D_48_min,D_48_max,D_48_last,D_49_mean,D_49_std,D_49_min,D_49_max,D_49_last,B_6_mean,B_6_std,B_6_min,B_6_max,B_6_last,B_7_mean,B_7_std,B_7_min,B_7_max,B_7_last,B_8_mean,B_8_std,B_8_min,B_8_max,B_8_last,D_50_mean,D_50_std,D_50_min,D_50_max,D_50_last,D_51_mean,D_51_std,D_51_min,D_51_max,D_51_last,B_9_mean,B_9_std,B_9_min,B_9_max,B_9_last,R_3_mean,R_3_std,R_3_min,R_3_max,R_3_last,D_52_mean,D_52_std,D_52_min,D_52_max,D_52_last,P_3_mean,P_3_std,P_3_min,P_3_max,P_3_last,B_10_mean,B_10_std,B_10_min,B_10_max,B_10_last,D_53_mean,D_53_std,D_53_min,D_53_max,D_53_last,S_5_mean,S_5_std,S_5_min,S_5_max,S_5_last,B_11_mean,B_11_std,B_11_min,B_11_max,B_11_last,S_6_mean,S_6_std,S_6_min,S_6_max,S_6_last,D_54_mean,D_54_std,D_54_min,D_54_max,D_54_last,R_4_mean,R_4_std,R_4_min,R_4_max,R_4_last,S_7_mean,S_7_std,S_7_min,S_7_max,S_7_last,B_12_mean,B_12_std,B_12_min,B_12_max,B_12_last,S_8_mean,S_8_std,S_8_min,S_8_max,S_8_last,D_55_mean,D_55_std,D_55_min,D_55_max,D_55_last,D_56_mean,D_56_std,D_56_min,D_56_max,D_56_last,B_13_mean,B_13_std,B_13_min,B_13_max,B_13_last,R_5_mean,R_5_std,R_5_min,R_5_max,R_5_last,D_58_mean,D_58_std,D_58_min,D_58_max,D_58_last,S_9_mean,S_9_std,S_9_min,S_9_max,S_9_last,B_14_mean,B_14_std,B_14_min,B_14_max,B_14_last,D_59_mean,D_59_std,D_59_min,D_59_max,D_59_last,D_60_mean,D_60_std,D_60_min,D_60_max,D_60_last,D_61_mean,D_61_std,D_61_min,D_61_max,D_61_last,B_15_mean,B_15_std,B_15_min,B_15_max,B_15_last,S_11_mean,S_11_std,S_11_min,S_11_max,S_11_last,D_62_mean,D_62_std,D_62_min,D_62_max,D_62_last,D_65_mean,D_65_std,D_65_min,D_65_max,D_65_last,B_16_mean,B_16_std,B_16_min,B_16_max,B_16_last,B_17_mean,B_17_std,B_17_min,B_17_max,B_17_last,B_18_mean,B_18_std,B_18_min,B_18_max,B_18_last,B_19_mean,B_19_std,B_19_min,B_19_max,B_19_last,B_20_mean,B_20_std,B_20_min,B_20_max,B_20_last,S_12_mean,S_12_std,S_12_min,S_12_max,S_12_last,R_6_mean,R_6_std,R_6_min,R_6_max,R_6_last,S_13_mean,S_13_std,S_13_min,S_13_max,S_13_last,B_21_mean,B_21_std,B_21_min,B_21_max,B_21_last,D_69_mean,D_69_std,D_69_min,D_69_max,D_69_last,B_22_mean,B_22_std,B_22_min,B_22_max,B_22_last,D_70_mean,D_70_std,D_70_min,D_70_max,D_70_last,D_71_mean,D_71_std,D_71_min,D_71_max,D_71_last,D_72_mean,D_72_std,D_72_min,D_72_max,D_72_last,S_15_mean,S_15_std,S_15_min,S_15_max,S_15_last,B_23_mean,B_23_std,B_23_min,B_23_max,B_23_last,D_73_mean,D_73_std,D_73_min,D_73_max,D_73_last,P_4_mean,P_4_std,P_4_min,P_4_max,P_4_last,D_74_mean,D_74_std,D_74_min,D_74_max,D_74_last,D_75_mean,D_75_std,D_75_min,D_75_max,D_75_last,D_76_mean,D_76_std,D_76_min,D_76_max,D_76_last,B_24_mean,B_24_std,B_24_min,B_24_max,B_24_last,R_7_mean,R_7_std,R_7_min,R_7_max,R_7_last,D_77_mean,D_77_std,D_77_min,D_77_max,D_77_last,B_25_mean,B_25_std,B_25_min,B_25_max,B_25_last,B_26_mean,B_26_std,B_26_min,B_26_max,B_26_last,D_78_mean,D_78_std,D_78_min,D_78_max,D_78_last,D_79_mean,D_79_std,D_79_min,D_79_max,D_79_last,R_8_mean,R_8_std,R_8_min,R_8_max,R_8_last,R_9_mean,R_9_std,R_9_min,R_9_max,R_9_last,S_16_mean,S_16_std,S_16_min,S_16_max,S_16_last,D_80_mean,D_80_std,D_80_min,D_80_max,D_80_last,R_10_mean,R_10_std,R_10_min,R_10_max,R_10_last,R_11_mean,R_11_std,R_11_min,R_11_max,R_11_last,B_27_mean,B_27_std,B_27_min,B_27_max,B_27_last,D_81_mean,D_81_std,D_81_min,D_81_max,D_81_las

In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,D_44_min,int8,0,0.00,27,"0, -1, 1, 2, 3, 4, 5, 6, 7, 8"
1,D_44_max,int8,0,0.00,32,"0, 1, 2, 3, 4, -1, 5, 6, 7, 8"
2,D_44_last,int8,0,0.00,31,"0, 1, 2, 3, -1, 4, 5, 6, 7, 8"
3,R_2_min,int8,0,0.00,2,"0, 1"
4,R_2_max,int8,0,0.00,2,"0, 1"
5,R_2_last,int8,0,0.00,2,"0, 1"
6,D_51_min,int8,0,0.00,8,"0, 1, 2, 3, 4, 5, 6, 7"
7,D_51_max,int8,0,0.00,9,"0, 1, 2, 3, 4, 5, 6, 7, 8"
8,D_51_last,int8,0,0.00,8,"0, 1, 2, 3, 4, 5, 6, 7"
9,R_3_min,int8,0,0.00,42,"0, 1, 2, 3, 4, 5, 6, 7, 8, 9"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
P_2_mean,458913.0,-0.929136,11.019608,-1.270000e+02,1.009993
P_2_std,453793.0,1.831865,9.666891,0.000000e+00,90.515991
P_2_min,458913.0,-4.569654,25.097486,-1.270000e+02,1.009993
P_2_max,458913.0,0.039117,9.279095,-1.270000e+02,1.010000
P_2_last,458913.0,-0.190504,10.236442,-1.270000e+02,1.009998
D_39_mean,458913.0,4.987931,5.605566,0.000000e+00,156.000000
D_39_std,453793.0,5.725742,5.302279,0.000000e+00,103.944697
D_39_min,458913.0,0.214001,2.092028,0.000000e+00,151.000000
D_39_max,458913.0,16.754258,16.100298,0.000000e+00,183.000000
D_39_last,458913.0,6.680637,13.672650,0.000000e+00,170.000000


In [7]:
# Categorical Feature Statistics
cat_stats

'No categorical/object features to summarize.'

In [8]:
# Target Distribution
target_df

,count,pct
target,,
0,340085,74.11
1,118828,25.89


## Task Curation

In [9]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from sklearn.model_selection import RepeatedStratifiedKFold

n_repeats, n_splits = 1, 3
sklearn_splits = RepeatedStratifiedKFold(
    n_repeats=n_repeats, n_splits=n_splits, random_state=42
).split(X=df.drop(columns=[task_mold.target_column_name]), y=df[task_mold.target_column_name])

splits = {}
for split_i, (train_idx, test_idx) in enumerate(sklearn_splits):
    repeat_i = split_i // n_splits
    fold_i = split_i % n_splits
    if repeat_i not in splits:
        splits[repeat_i] = {}
    splits[repeat_i][fold_i] = (train_idx.tolist(), test_idx.tolist())

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Stratified 3-fold cross-validation following our default recommendations.",
    splits=splits,
)

## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
019bc7b9-1e0b-7253-869b-9e4ade870b35
9e84fd9337cc22f61bded997818f2883b6e94681607a275f51b7ef4f66721efc
